## Imports

In [1]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras_tuner as kt
from datetime import datetime
import pickle
import os

In [2]:
log_dir = './my_logs/run_1'

## Loading Data

In [3]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print('Train shapes : ',x_train_full.shape, y_train_full.shape)
print('Test shapes : ',x_test.shape, y_test.shape)

Train shapes :  (60000, 28, 28) (60000,)
Test shapes :  (10000, 28, 28) (10000,)


In [4]:
x_train, y_train = x_train_full[:-5000] / 255., y_train_full[:-5000]
x_valid, y_valid = x_train_full[-5000:] / 255., y_train_full[-5000:]
x_test = x_test / 255.

In [5]:
print('Train shapes : ',x_train.shape, y_train.shape)
print('Validation shapes : ',x_valid.shape, y_valid.shape)

Train shapes :  (55000, 28, 28) (55000,)
Validation shapes :  (5000, 28, 28) (5000,)


---

## Finding the optimal learning rate

In [6]:
tf.random.set_seed(42)
model = tf.keras.Sequential()
model.add(tf.keras.layers.Input(shape=(28, 28)))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(300, activation='relu'))
model.add(tf.keras.layers.Dense(300, activation='relu'))
model.add(tf.keras.layers.Dense(10, activation='softmax'))

tensorboard_cb = tf.keras.callbacks.TensorBoard(log_dir=log_dir)

model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=1e-10), loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [7]:
# Slowly increase lr, find where val loss starts jumping, optimal lr is 10 times smaller
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lambda epoch, lr: lr*10 if (epoch+1)%5==0 else lr)

In [8]:
history = model.fit(x_train, y_train, epochs=50, validation_data=(x_valid, y_valid), callbacks=[tensorboard_cb, lr_scheduler])

Epoch 1/50
1719/1719 [==============================] - 9s 5ms/step - loss: 2.3936 - accuracy: 0.0872 - val_loss: 2.3986 - val_accuracy: 0.0818 - lr: 1.0000e-10
Epoch 2/50
1719/1719 [==============================] - 7s 4ms/step - loss: 2.3936 - accuracy: 0.0872 - val_loss: 2.3986 - val_accuracy: 0.0818 - lr: 1.0000e-10
Epoch 3/50
1719/1719 [==============================] - 7s 4ms/step - loss: 2.3936 - accuracy: 0.0872 - val_loss: 2.3986 - val_accuracy: 0.0818 - lr: 1.0000e-10
Epoch 4/50
1719/1719 [==============================] - 7s 4ms/step - loss: 2.3936 - accuracy: 0.0872 - val_loss: 2.3986 - val_accuracy: 0.0818 - lr: 1.0000e-10
Epoch 5/50
1719/1719 [==============================] - 7s 4ms/step - loss: 2.3936 - accuracy: 0.0872 - val_loss: 2.3986 - val_accuracy: 0.0818 - lr: 1.0000e-09
Epoch 6/50
1719/1719 [==============================] - 7s 4ms/step - loss: 2.3936 - accuracy: 0.0872 - val_loss: 2.3986 - val_accuracy: 0.0818 - lr: 1.0000e-09
Epoch 7/50
1719/1719 [============

In [9]:
model.evaluate(x_test, y_test)

313/313 [==============================] - 1s 3ms/step - loss: 2.3161 - accuracy: 0.0958


[2.3160557746887207, 0.0957999974489212]

In [3]:
# pickle.dump(history.history, open(os.path.join(log_dir, 'history.pkl'), 'wb'))

## Hyperparmeter Tuning

In [6]:
history = pickle.load(open(os.path.join(log_dir, 'history.pkl'), 'rb'))

In [7]:
df_history = pd.DataFrame(history)
df_history.tail()

,loss,accuracy,val_loss,val_accuracy,lr
45,9.024342e-02,0.971982,0.094056,0.9744,0.1
46,6.398229e-02,0.980582,0.086967,0.9726,0.1
47,4.831813e-02,0.984600,0.069723,0.9804,0.1
48,3.440611e-02,0.989073,0.074977,0.9788,0.1
49,1.345031e+15,0.106655,2.312479,0.0964,1.0


In [8]:
best_lr = 0.1/10 # loss shoots up at 0.1, so 10 times smaller is optimal  

In [9]:
class MyHyperModelClass(kt.HyperModel):
    def build(self, hp):
        n_hidden = hp.Int('n_hidden', min_value=1, max_value=5, step=1)
        n_neurons = hp.Int('n_neurons', min_value=50, max_value=500, step=10)
        optimizer = hp.Choice('optimizer', values=['adam', 'sgd'])
        model = tf.keras.Sequential()
        model.add(tf.keras.layers.Input(shape=(28, 28)))
        model.add(tf.keras.layers.Flatten())
        for _ in range(n_hidden):
            model.add(tf.keras.layers.Dense(n_neurons, activation='relu'))
        model.add(tf.keras.layers.Dense(10, activation='softmax'))
        if optimizer == 'sgd':
            model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=best_lr), loss='sparse_categorical_crossentropy',metrics=['accuracy'])
        else:
            model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=best_lr), loss='sparse_categorical_crossentropy',metrics=['accuracy'])
        return model

In [10]:
bayesian_opt_tuner = kt.BayesianOptimization(hypermodel=MyHyperModelClass(), objective='val_accuracy', seed=42, max_trials=10, overwrite=True, directory=log_dir, project_name='mnist')

In [11]:
tensorboard_cb = tf.keras.callbacks.TensorBoard(log_dir=log_dir)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(monitor='val_loss' ,patience=5, restore_best_weights=True)

In [12]:
bayesian_opt_tuner.search(x_train, y_train, epochs=50, validation_data=(x_valid, y_valid), callbacks=[tensorboard_cb, early_stopping_cb])

Trial 10 Complete [00h 04m 54s]
val_accuracy: 0.9801999926567078

Best val_accuracy So Far: 0.9842000007629395
Total elapsed time: 00h 39m 54s
INFO:tensorflow:Oracle triggered exit


In [16]:
bayesian_opt_tuner.get_best_models()[0].evaluate(x_valid, y_valid)

157/157 [==============================] - 1s 4ms/step - loss: 0.0650 - accuracy: 0.9832


[0.06495922803878784, 0.9832000136375427]

In [ ]:
bayesian_opt_tuner.get_best_models()[0].evaluate(x_test, y_test)

313/313 [==============================] - 1s 3ms/step - loss: 0.0646 - accuracy: 0.9800


[0.06462358683347702, 0.9800000190734863]

In [33]:
%load_ext tensorboard

In [34]:
%tensorboard --logdir=./my_logs